# Instabee locker network agent

An AI agent that answers locker-network questions — *"where should the next Instabox lockers go?"* — from public data:
OpenStreetMap lockers, SCB population, Overture places and buildings, and Paketindex market figures, joined on an
H3 grid in `data/lockers.duckdb`.

It follows the data-agent workshop pattern: a read-only DuckDB warehouse, schema tools, a data contract in the
system prompt, and an eval suite. The difference is that coverage, demand and site choice are **tested Python
tools**, so the model never does the maths itself.

## Part 1 — Set up

In [ ]:
from IPython.display import Markdown, IFrame, display
from agent import analysis, maps, tools
from agent.client import make_client
from agent.loop import LockerAgent
from agent.__main__ import _trace
from pipeline.common import ROOT, connect

con = connect(read_only=True)          # read-only at the connection level, as in the workshop
print(tools.list_tables(con))

## Part 2 — The tools on their own (no model, no API key needed)

Every number the agent reports comes from one of these functions. Run them directly to see what the model sees.

In [ ]:
print(tools.run_tool(con, "coverage", {"area": "Uppsala", "network": "Instabee", "radius_m": 500})[0])

In [ ]:
print(tools.run_tool(con, "coverage", {"area": "Sweden", "network": "any", "group_by": "kommun", "min_population": 50000})[0])

In [ ]:
print(tools.run_tool(con, "optimise_next_n", {"area": "Uppsala", "n": 10})[0])

## Part 3 — Ask the agent

Needs `ANTHROPIC_API_KEY` in `.env` (plus `ANTHROPIC_WORKSPACE_ID` for a user-scoped key).
The trace shows each tool call, so you can check the agent's working, not only its answer.

In [ ]:
agent = LockerAgent(make_client(), con, effort="medium")

def ask(question):
    result = agent.ask(question, on_event=_trace)
    display(Markdown(result.answer))
    print(f"{len(result.tool_calls)} tool calls · {result.seconds}s · {result.usage}")
    return result

In [ ]:
r = ask("Where should Instabox put its next 10 lockers in Uppsala?")

Try also:
- Which Swedish cities with over 50,000 residents are least covered by parcel lockers?
- Which parts of Göteborg have the most residents without an Instabee locker within 300 m?
- Which grocery chains already host PostNord lockers but not Instabee ones?
- How many compartments would a site with 20,000 locker parcels a year need in December?
- What is the utilisation of our Uppsala lockers? *(the agent should say public data cannot answer this)*

## Part 4 — Map the recommendation

In [ ]:
result = analysis.optimise_next_n(con, "Uppsala", n=10)
sites = [{"name": f"{row.rank}. {row.name}", "lat": row.lat, "lon": row.lon} for row in result.table.itertuples()]
path = maps.render_map(con, "Uppsala", sites=sites)
print(path)
display(IFrame(src=path.relative_to(ROOT).as_posix(), width="100%", height=520))

## Part 5 — Evaluate

Each question has a reference answer computed from the data at run time. An LLM judge grades the answer, and a
faithfulness check flags any number in the answer that no tool returned. Run the full suite from a terminal with
`python -m evals.run_evals`.

In [ ]:
from evals.run_evals import main as run_evals
run_evals(["--only", "coverage"])